# Experiment: TF-IDF Baseline Model Sweep

Objective: run a small, reproducible hyperparameter sweep for TF-IDF + Logistic Regression,
select the best setting by macro-F1, then train a final model and generate `submission.csv`.


## 1. Setup


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.json"
TEST_PATH = DATA_DIR / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for target in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR, MODELS_DIR]:
    target.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.data import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")


## 4. Data Validation


In [ ]:
assert train_df[REVIEW_COL].str.len().gt(0).all(), "Found empty review text in training set"
assert train_df[LABEL_COL].isin([0, 1]).all(), "Found labels outside {0,1}"

label_ratio = (train_df[LABEL_COL].value_counts(normalize=True) * 100).round(2)
label_ratio


## 5. Exploratory Analysis


In [ ]:
train_df["review_word_len"] = train_df[REVIEW_COL].str.split().str.len()
summary = train_df.groupby(LABEL_COL)["review_word_len"].describe().round(2)
summary


## 6. Feature Engineering


In [ ]:
search_space = [
    {"name": "uni_bi_50k_c4", "ngram_min": 1, "ngram_max": 2, "max_features": 50_000, "min_df": 2, "regularization_c": 4.0},
    {"name": "uni_bi_100k_c4", "ngram_min": 1, "ngram_max": 2, "max_features": 100_000, "min_df": 2, "regularization_c": 4.0},
    {"name": "uni_tri_80k_c6", "ngram_min": 1, "ngram_max": 3, "max_features": 80_000, "min_df": 2, "regularization_c": 6.0},
    {"name": "bi_tri_120k_c3", "ngram_min": 2, "ngram_max": 3, "max_features": 120_000, "min_df": 2, "regularization_c": 3.0},
]

pd.DataFrame(search_space)


## 7. Modeling


In [ ]:
from sentiment_project.training import train_and_evaluate

runs = []
best_model = None
best_metrics = None
best_config = None
best_macro_f1 = -1.0

for cfg in search_space:
    model, metrics = train_and_evaluate(
        train_df,
        test_size=0.2,
        random_state=RANDOM_SEED,
        ngram_min=cfg["ngram_min"],
        ngram_max=cfg["ngram_max"],
        max_features=cfg["max_features"],
        min_df=cfg["min_df"],
        regularization_c=cfg["regularization_c"],
    )
    run = {
        "name": cfg["name"],
        "ngram_range": f"({cfg['ngram_min']}, {cfg['ngram_max']})",
        "max_features": cfg["max_features"],
        "regularization_c": cfg["regularization_c"],
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
    }
    runs.append(run)

    if metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = metrics["macro_f1"]
        best_model = model
        best_metrics = metrics
        best_config = cfg

results_df = pd.DataFrame(runs).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results_df


## 8. Evaluation


In [ ]:
print("Best config:", best_config)
print("Best validation accuracy:", round(best_metrics["accuracy"], 5))
print("Best validation macro_f1:", round(best_metrics["macro_f1"], 5))

ax = sns.barplot(data=results_df, x="name", y="macro_f1")
ax.set_title("Validation macro-F1 across TF-IDF settings")
ax.set_xlabel("Configuration")
ax.set_ylabel("macro-F1")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 9. Inference / Submission


In [ ]:
import joblib

from sentiment_project.inference import build_submission_dataframe, save_submission_csv
from sentiment_project.modeling import build_tfidf_logreg_pipeline

final_model = build_tfidf_logreg_pipeline(
    ngram_min=best_config["ngram_min"],
    ngram_max=best_config["ngram_max"],
    max_features=best_config["max_features"],
    min_df=best_config["min_df"],
    regularization_c=best_config["regularization_c"],
    random_state=RANDOM_SEED,
)
final_model.fit(train_df[REVIEW_COL], train_df[LABEL_COL])

model_path = MODELS_DIR / "tfidf_logreg_notebook.joblib"
joblib.dump(final_model, model_path)

submission_path = SUBMISSION_DIR / "submission.csv"
preds = final_model.predict(test_df[REVIEW_COL])
submission_df = build_submission_dataframe(preds, include_id=True)
save_submission_csv(submission_df, submission_path)

metrics_path = REPORTS_DIR / "train_metrics_notebook.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "best_config": best_config,
            "best_validation": {
                "accuracy": best_metrics["accuracy"],
                "macro_f1": best_metrics["macro_f1"],
            },
            "runs": runs,
        },
        f,
        indent=2,
    )

print(f"Saved model: {model_path.relative_to(PROJECT_ROOT)}")
print(f"Saved metrics: {metrics_path.relative_to(PROJECT_ROOT)}")
print(f"Saved submission: {submission_path.relative_to(PROJECT_ROOT)}")
submission_df.head()


## 10. Conclusions / Next Steps

- The sweep notebook compares several TF-IDF feature settings with a fixed deterministic split.
- Best-performing configuration is selected by validation macro-F1.
- Final model is retrained on full training data and used to generate `data/submissions/submission.csv`.

Next:
1. Add cross-validation to reduce split sensitivity.
2. Add qualitative error analysis against selected prediction samples.
3. Benchmark against a transformer baseline to improve minority-class F1.
